In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
# imports
from voxforge.data.load import load_all
from voxforge.data.config import INTERIM_DATA_DIR, REPORTS_DIR
from voxforge.data.merge import merge_datasets
from voxforge.data.profile import dataset_overview, text_statistics, vocabulary_size, data_quality_report,generate_profile

import json
from pathlib import Path

In [3]:
# load all the 3 datasets
datasets = load_all()
df_merged = merge_datasets(datasets)

df_merged.shape

INFO - Loaded 34660 rows
INFO - Loaded 5000 rows
INFO - Loaded 28332 rows
INFO - Loaded 3 datasets.


(67992, 27)

In [4]:
# saved the merged the 3 raw dataset under /data/processed
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

output_path = INTERIM_DATA_DIR / "merged_reviews.csv"

df_merged.to_csv(
    output_path,
    index=False,
)

print(f"Merged dataset saved to: {output_path}")

Merged dataset saved to: /Users/karima/Ironhack-challenges/voxforge-ai-review-analytics/data/interim/merged_reviews.csv


In [5]:
# datasset overview
dataset_overview(df_merged)

,Metric,Value
0,Rows,67992.00
1,Columns,27.00
2,Duplicate Rows,95.00
3,Memory Usage (MB),458.82
4,Total Missing Values,555201.00


In [6]:
# Column information
df_merged.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 67992 entries, 0 to 67991
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    67992 non-null  str    
 1   name                  61232 non-null  str    
 2   asins                 67990 non-null  str    
 3   brand                 67992 non-null  str    
 4   categories            67992 non-null  str    
 5   keys                  67992 non-null  str    
 6   manufacturer          67992 non-null  str    
 7   reviews.date          67953 non-null  str    
 8   reviews.dateAdded     25091 non-null  str    
 9   reviews.dateSeen      67992 non-null  str    
 10  reviews.didPurchase   10 non-null     object 
 11  reviews.doRecommend   55152 non-null  object 
 12  reviews.id            71 non-null     float64
 13  reviews.numHelpful    55246 non-null  float64
 14  reviews.rating        67959 non-null  float64
 15  reviews.sourceURLs    67992 no

In [7]:
# Check missing values
df_merged.isna().sum()

id                          0
name                     6760
asins                       2
brand                       0
categories                  0
keys                        0
manufacturer                0
reviews.date               39
reviews.dateAdded       42901
reviews.dateSeen            0
reviews.didPurchase     67982
reviews.doRecommend     12840
reviews.id              67921
reviews.numHelpful      12746
reviews.rating             33
reviews.sourceURLs          0
reviews.text                1
reviews.title              19
reviews.userCity        67992
reviews.userProvince    67992
reviews.username           13
dateAdded               34660
dateUpdated             34660
primaryCategories       34660
imageURLs               34660
manufacturerNumber      34660
sourceURLs              34660
dtype: int64

In [8]:
# Analyze duplicates
df_merged.duplicated().sum()

np.int64(95)

In [10]:
# check unique values
display(
    df_merged["reviews.rating"]
    .value_counts(dropna=False)
    .sort_index()
)

print("Unique category strings:", df_merged["categories"].nunique())

reviews.rating
1.0     1438
2.0     1072
3.0     2902
4.0    15397
5.0    47150
NaN       33
Name: count, dtype: int64

Unique category strings: 111


In [ ]:
# create numerical summary 
df_merged.describe()

,reviews.id,reviews.numHelpful,reviews.rating,reviews.userCity,reviews.userProvince
count,7.100000e+01,55246.000000,67959.000000,0.0,0.0
mean,1.837463e+08,0.572041,4.556071,NaN,NaN
std,2.371858e+07,11.587020,0.825126,NaN,NaN
min,1.082112e+08,0.000000,1.000000,NaN,NaN
25%,1.843760e+08,0.000000,4.000000,NaN,NaN
50%,1.880757e+08,0.000000,5.000000,NaN,NaN
75%,1.987126e+08,0.000000,5.000000,NaN,NaN
max,2.085304e+08,814.000000,5.000000,NaN,NaN


In [ ]:
# text profiling
text_statistics(df_merged)

,Metric,Value
0,Average Characters,150.24
1,Median Characters,99.00
2,Minimum Characters,0.00
3,Maximum Characters,10670.00
4,Average Words,28.59
5,Median Words,19.00
6,Minimum Words,0.00
7,Maximum Words,1858.00


In [ ]:
# calculate vocabulary size
vocabulary_size(df_merged)

39937

In [ ]:
# generate data quality overview
report = data_quality_report(df_merged)

report

{'rows': 67992,
 'columns': 27,
 'duplicates': 95,
 'missing_values': {'id': 0,
  'name': 6760,
  'asins': 2,
  'brand': 0,
  'categories': 0,
  'keys': 0,
  'manufacturer': 0,
  'reviews.date': 39,
  'reviews.dateAdded': 42901,
  'reviews.dateSeen': 0,
  'reviews.didPurchase': 67982,
  'reviews.doRecommend': 12840,
  'reviews.id': 67921,
  'reviews.numHelpful': 12746,
  'reviews.rating': 33,
  'reviews.sourceURLs': 0,
  'reviews.text': 1,
  'reviews.title': 19,
  'reviews.userCity': 67992,
  'reviews.userProvince': 67992,
  'reviews.username': 13,
  'dateAdded': 34660,
  'dateUpdated': 34660,
  'primaryCategories': 34660,
  'imageURLs': 34660,
  'manufacturerNumber': 34660,
  'sourceURLs': 34660},
 'data_types': {'id': 'str',
  'name': 'str',
  'asins': 'str',
  'brand': 'str',
  'categories': 'str',
  'keys': 'str',
  'manufacturer': 'str',
  'reviews.date': 'str',
  'reviews.dateAdded': 'str',
  'reviews.dateSeen': 'str',
  'reviews.didPurchase': 'object',
  'reviews.doRecommend': '

In [12]:
# Export data quality report
profile = generate_profile(df_merged)

with open(
    REPORTS_DIR / "data_profile.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        profile["quality_report"],
        f,
        indent=4,
        ensure_ascii=False,
    )

In [14]:
# geneate the 3 data profiles
profile = generate_profile(df_merged)

display(profile["overview"])
display(profile["text_statistics"])
profile["quality_report"]

,Metric,Value
0,Rows,67992.00
1,Columns,27.00
2,Duplicate Rows,95.00
3,Memory Usage (MB),458.82
4,Total Missing Values,555201.00


,Metric,Value
0,Average Characters,150.24
1,Median Characters,99.00
2,Minimum Characters,0.00
3,Maximum Characters,10670.00
4,Average Words,28.59
5,Median Words,19.00
6,Minimum Words,0.00
7,Maximum Words,1858.00


{'rows': 67992,
 'columns': 27,
 'duplicates': 95,
 'missing_values': {'id': 0,
  'name': 6760,
  'asins': 2,
  'brand': 0,
  'categories': 0,
  'keys': 0,
  'manufacturer': 0,
  'reviews.date': 39,
  'reviews.dateAdded': 42901,
  'reviews.dateSeen': 0,
  'reviews.didPurchase': 67982,
  'reviews.doRecommend': 12840,
  'reviews.id': 67921,
  'reviews.numHelpful': 12746,
  'reviews.rating': 33,
  'reviews.sourceURLs': 0,
  'reviews.text': 1,
  'reviews.title': 19,
  'reviews.userCity': 67992,
  'reviews.userProvince': 67992,
  'reviews.username': 13,
  'dateAdded': 34660,
  'dateUpdated': 34660,
  'primaryCategories': 34660,
  'imageURLs': 34660,
  'manufacturerNumber': 34660,
  'sourceURLs': 34660},
 'data_types': {'id': 'str',
  'name': 'str',
  'asins': 'str',
  'brand': 'str',
  'categories': 'str',
  'keys': 'str',
  'manufacturer': 'str',
  'reviews.date': 'str',
  'reviews.dateAdded': 'str',
  'reviews.dateSeen': 'str',
  'reviews.didPurchase': 'object',
  'reviews.doRecommend': '